# 02 — Knowledge Base Build

## 0. Imports & paths

In [1]:
import sys
from pathlib import Path

# Make sure the project root is on the Python path
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CLAUSES_PATH        = PROJECT_ROOT / "data/processed/clauses.jsonl"
EMBEDDINGS_PATH     = PROJECT_ROOT / "data/processed/clause_embeddings.npy"
BM25_INDEX_PATH     = PROJECT_ROOT / "data/processed/bm25_index.pkl"
QDRANT_PERSIST_PATH = PROJECT_ROOT / "data/processed/qdrant_local"

print(f"Project root : {PROJECT_ROOT}")
print(f"Clauses file : {CLAUSES_PATH}  (exists={CLAUSES_PATH.exists()})")

Project root : /Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot
Clauses file : /Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot/data/processed/clauses.jsonl  (exists=True)


## 1. Load preprocessed clauses

In [2]:
from src.retrieval.embedder import load_clauses

clauses = load_clauses(CLAUSES_PATH)

print(f"Total clauses loaded : {len(clauses):,}")
print(f"\n--- Sample clause record ---")
import json
sample = clauses[500]
print(json.dumps({k: v if k != 'clause_text' else v[:300] + '...' for k, v in sample.items()}, indent=2))

2026-04-10 11:36:05,507 | INFO | Loaded 23564 clause records from /Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot/data/processed/clauses.jsonl


Total clauses loaded : 23,564

--- Sample clause record ---
{
  "split": "train",
  "contract_id": "train_00016",
  "clause_id": "train_00016_clause_044",
  "clause_index": 44,
  "num_clauses": 98,
  "char_count": 230,
  "clause_text": "4.6.1. \"VerticalNet Impresse Users\" are the Users that register with\nImpresse through the Co-Branded Site, but specifically excluding those Users\nwho have previously registered with Impresse other than through the\nCo-Branded Site...."
}


## 2. Embed clauses → dense vectors

- **Model used**: `sentence-transformers/all-MiniLM-L6-v2` (384-dim, fast, good quality)
- If you have a GPU or want the legal-domain model, change to: `nlpaueb/legal-bert-base-uncased`  
  *(Note: legal-bert needs `--pooling cls` — the embedder handles this automatically)*

**Cached**: If `clause_embeddings.npy` already exists on disk, we skip re-embedding.

In [3]:
import numpy as np
from src.retrieval.embedder import ClauseEmbedder

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

if EMBEDDINGS_PATH.exists():
    print(f"Loading cached embeddings from {EMBEDDINGS_PATH}…")
    embeddings = np.load(str(EMBEDDINGS_PATH))
    print(f"Loaded embeddings shape: {embeddings.shape}")
else:
    embedder = ClauseEmbedder(model_name=MODEL_NAME, batch_size=64)
    embeddings = embedder.embed(clauses)

    # Save to disk so we don't have to re-embed every run
    np.save(str(EMBEDDINGS_PATH), embeddings)
    print(f"\nSaved embeddings → {EMBEDDINGS_PATH}")
    print(f"Shape: {embeddings.shape}  |  dtype: {embeddings.dtype}")

Loading cached embeddings from /Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot/data/processed/clause_embeddings.npy…
Loaded embeddings shape: (23564, 384)


In [4]:
# Quick sanity check
assert len(clauses) == len(embeddings), "Mismatch between clauses and embeddings!"
print(f"✅ {len(clauses):,} clauses  ↔  {embeddings.shape} embedding matrix")
print(f"   L2 norm of first vector: {np.linalg.norm(embeddings[0]):.4f}  (should be ~1.0, we normalize)")

✅ 23,564 clauses  ↔  (23564, 384) embedding matrix
   L2 norm of first vector: 1.0000  (should be ~1.0, we normalize)


## 3. Store in Qdrant Vector DB

In [5]:
from src.retrieval.vector_store import QdrantVectorStore

# Use on-disk persistence so the KB survives notebook restarts
store = QdrantVectorStore(
    persist_path=QDRANT_PERSIST_PATH,
    embedding_dim=embeddings.shape[1],   # 384 for MiniLM
)

# Create collection (skip if it already exists)
store.create_collection(reset=False)
print("Collection created / verified.")

2026-04-10 11:36:16,037 | INFO | Using on-disk Qdrant at /Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot/data/processed/qdrant_local
2026-04-10 11:36:16,049 | INFO | Created collection 'contractsense_clauses'  (dim=384, metric=cosine)


Collection created / verified.


In [6]:
# Check if already populated
try:
    count = store.count()
    print(f"Points already in collection: {count:,}")
    if count == len(clauses):
        print("✅ Collection already fully populated — skipping upsert.")
        already_done = True
    else:
        already_done = False
except Exception:
    already_done = False
    print("Collection is empty — will upsert.")

Points already in collection: 0


In [7]:
if not already_done:
    print(f"Upserting {len(clauses):,} clause vectors into Qdrant…")
    store.upsert(clauses, embeddings, batch_size=256)
    print(f"\n✅ Done! Points in collection: {store.count():,}")

Upserting 23,564 clause vectors into Qdrant…


2026-04-10 11:36:16,558 | INFO | Upserted 256 / 23564 points
2026-04-10 11:36:18,461 | INFO | Upserted 2816 / 23564 points
2026-04-10 11:36:20,129 | INFO | Upserted 5376 / 23564 points
2026-04-10 11:36:21,780 | INFO | Upserted 7936 / 23564 points
2026-04-10 11:36:23,315 | INFO | Upserted 10496 / 23564 points
2026-04-10 11:36:24,906 | INFO | Upserted 13056 / 23564 points
2026-04-10 11:36:26,485 | INFO | Upserted 15616 / 23564 points
2026-04-10 11:36:28,172 | INFO | Upserted 18176 / 23564 points
/Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot/src/retrieval/vector_store.py:171: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20224 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client.upsert(
2026-04-10 11:36:29,688 | INFO | Upserted 20736 / 23564 points
2026-04-10 11:36:31,313 | INFO | Upserted 23296 / 23564 points
2026-04-10 11:36:31,


✅ Done! Points in collection: 23,564


In [8]:
# Collection info
import json
info = store.collection_info()
print(json.dumps(info, indent=2, default=str))

{
  "name": "contractsense_clauses",
  "points_count": 23564,
  "vector_size": 384,
  "distance_metric": "Cosine",
  "status": "green"
}


## 4. Test Dense Retrieval (Qdrant)

In [9]:
from src.retrieval.embedder import ClauseEmbedder

# Reuse embedder (or create a new one if kernel restarted)
try:
    _ = embedder
except NameError:
    embedder = ClauseEmbedder(model_name=MODEL_NAME)

def dense_search(query: str, top_k: int = 5):
    qvec = embedder.embed_query(query)
    return store.search(qvec, top_k=top_k)

# --- Test queries ---
queries = [
    "auto-renewal clause",
    "indemnification liability cap",
    "termination for convenience",
    "governing law jurisdiction",
]

for q in queries:
    results = dense_search(q, top_k=3)
    print(f"\n🔍 Query: '{q}'")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] score={r['score']:.4f}  contract={r['contract_id']}")
        print(f"      {r['clause_text'][:200]}…")

2026-04-10 11:36:31,535 | INFO | Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-04-10 11:36:31,538 | INFO | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-04-10 11:36:40,642 | INFO | Use pytorch device_name: mps
2026-04-10 11:36:40,984 | INFO | Embedding dimension: 384


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Query: 'auto-renewal clause'
  [1] score=0.7404  contract=train_00023
      8.1. Automatic Renewal. This Agreement will automatically renew at the end
of the Initial Term or a subsequent renewal term on a year to year basis (each,
a "Renewal Term"), unless either party notifi…
  [2] score=0.6981  contract=train_00023
      1.26. Renewal Term shall have the meaning ascribed thereto in Section 8.1 [Automatic Renewal].…
  [3] score=0.6439  contract=train_00491
      24.2 Renewal. Unless this Agreement is terminated pursuant to Section 25, this Agreement will automatically renew for additional successive [***]
terms (each a “Renewal Term” and together with the Ini…


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Query: 'indemnification liability cap'
  [1] score=0.6917  contract=train_00500
      9.1.1 The Liability Cap will not apply to any (1) material confidentiality breach under Section 5, and/or (2) indemnification
obligations under Section 7.1.…
  [2] score=0.6600  contract=train_00321
      11.3. Liability Cap. Without affecting Section 10 or the respective obligations of the parties under the Confidentiality Agreement and except
for any liability (i) relating to any breach associated wi…
  [3] score=0.6457  contract=train_00167
      13.4 No Consequential Damages and Limitation of Liability 33
Article
14 Indemnification 33…


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Query: 'termination for convenience'
  [1] score=0.6717  contract=train_00057
      12.2 Termination for Convenience. This Agreement may be terminated by either party for any reason or no reason, whether
or not extended beyond the initial term, by giving the other party written notic…
  [2] score=0.6708  contract=train_00419
      16.4 Effect of Termination. The termination or expiration of this Agreement shall not release either of the Parties from any liability which at the
time of termination or expiry has already accrued to…
  [3] score=0.6598  contract=train_00011
      Section 6.03 [Effect of Termination]. Effect of Termination. In the event of the termination of
this Agreement, all rights and obligations of PrimeCall and DeltaThree shall
terminate as of the effecti…


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Query: 'governing law jurisdiction'
  [1] score=0.6717  contract=train_00043
      9.10. Governing Law; Jurisdiction. This Agreement shall be governed in accordance with the laws of the State of New York
without regard to conflict of laws principles. It is hereby irrevocably agreed …
  [2] score=0.6582  contract=train_00219
      9.6 Governing Law; Jurisdiction. This Agreement shall be interpreted, construed, governed, and enforced according to the laws of the
Commonwealth of Virginia, without giving effect to its conflict of …
  [3] score=0.6458  contract=train_00297
      5.6. Governing Law; Jurisdiction. This Agreement and all related business
transactions will be governed by the laws of the Commonwealth of Massachusetts
(without reference to principles of conflicts o…


## 5. Build BM25 Baseline Retriever

In [10]:
from src.retrieval.bm25_retriever import BM25Retriever, build_bm25_from_jsonl

if BM25_INDEX_PATH.exists():
    print(f"Loading cached BM25 index from {BM25_INDEX_PATH}…")
    bm25 = BM25Retriever.load(BM25_INDEX_PATH)
else:
    print("Building BM25 index from clauses JSONL…")
    bm25 = build_bm25_from_jsonl(
        clauses_path=CLAUSES_PATH,
        save_index_path=BM25_INDEX_PATH,
    )
    print(f"✅ BM25 index saved to {BM25_INDEX_PATH}")

2026-04-10 11:36:48,350 | INFO | Loaded 23564 clauses from /Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot/data/processed/clauses.jsonl
2026-04-10 11:36:48,351 | INFO | Tokenizing 23564 clauses for BM25 index…


Building BM25 index from clauses JSONL…


2026-04-10 11:36:50,675 | INFO | BM25 index built successfully.
2026-04-10 11:36:51,212 | INFO | BM25 retriever saved to /Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot/data/processed/bm25_index.pkl


✅ BM25 index saved to /Users/sanjananathani/Desktop/Projects/ContractSense/ContractSense-copilot/data/processed/bm25_index.pkl


In [11]:
# --- Test BM25 ---
for q in queries:
    results = bm25.search(q, top_k=3)
    print(f"\n📚 Query: '{q}'")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] bm25_score={r['bm25_score']:.4f}  contract={r['contract_id']}")
        print(f"      {r['clause_text'][:200]}…")


📚 Query: 'auto-renewal clause'
  [1] bm25_score=7.8864  contract=train_00287
      30.1 The Contract hereof is perfected with the subscription of the same. For its execution the approval of the bond assumed by the SENDER is
required.
16
ANNEX 1
MANUAL FOR THE TRANSPORTER OF PIPELINE…
  [2] bm25_score=7.7252  contract=train_00257
      9.2 Nothing in this Agreement shall exclude or limit either party’s liability under Clause 10 (Indemnities), or Distributor’s liability under Clause
2 (License Grants and Restrictions), Clause 3.5 (Ex…
  [3] bm25_score=7.2904  contract=train_00498
      21.13.1If any Clause, or part of a Clause, of this Agreement, is found by any court or administrative body of competent jurisdiction to be illegal,
invalid or unenforceable, and the provision in quest…

📚 Query: 'indemnification liability cap'
  [1] bm25_score=19.9237  contract=train_00500
      9.1.1 The Liability Cap will not apply to any (1) material confidentiality breach under Section 5, and/or (2) i

## 6. Side-by-Side: Dense vs BM25

This is your **baseline comparison** — key for the final report.

In [12]:
import textwrap

def compare_retrievers(query: str, top_k: int = 3):
    """Print Dense and BM25 results side-by-side for a query."""
    print("=" * 80)
    print(f"QUERY: '{query}'")
    print("=" * 80)

    dense_results = dense_search(query, top_k=top_k)
    bm25_results  = bm25.search(query, top_k=top_k)

    print(f"\n{'DENSE (Qdrant / MiniLM)':<40} {'BM25 (Okapi)':<40}")
    print("-" * 80)

    for i in range(top_k):
        d = dense_results[i] if i < len(dense_results) else {}
        b = bm25_results[i]  if i < len(bm25_results)  else {}

        d_text = textwrap.shorten(d.get('clause_text', ''), width=36)
        b_text = textwrap.shorten(b.get('clause_text', ''), width=36)

        d_score = f"score={d.get('score', 0):.4f}"
        b_score = f"score={b.get('bm25_score', 0):.4f}"

        print(f"  [{i+1}] {d_score:<15} {d_text:<30} | [{i+1}] {b_score:<15} {b_text:<30}")

    # Key insight: synonyms test
    dense_ids = {r.get('clause_id') for r in dense_results}
    bm25_ids  = {r.get('clause_id') for r in bm25_results}
    overlap   = len(dense_ids & bm25_ids)
    print(f"\n  Overlap in Top-{top_k}: {overlap} / {top_k} clauses")

    print()

# Synonym test — BM25 will fail here, dense should succeed
compare_retrievers("evergreen provision automatic extension")  # synonym for auto-renewal
compare_retrievers("indemnification clause liability cap")
compare_retrievers("termination for convenience notice period")

QUERY: 'evergreen provision automatic extension'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


DENSE (Qdrant / MiniLM)                  BM25 (Okapi)                            
--------------------------------------------------------------------------------
  [1] score=0.3815    8.1. Automatic Renewal. This [...] | [1] score=15.9640   2.1Term. This Agreement shall [...]
  [2] score=0.3778    9.1 In accordance with items [...] | [2] score=14.6962   17.2 The Agreement may be [...]
  [3] score=0.3752    3.2 Renewal Each Service [...] | [3] score=11.5107   3.1 Term. (a) The term of this [...]

  Overlap in Top-3: 0 / 3 clauses

QUERY: 'indemnification clause liability cap'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


DENSE (Qdrant / MiniLM)                  BM25 (Okapi)                            
--------------------------------------------------------------------------------
  [1] score=0.7063    9.1.1 The Liability Cap will [...] | [1] score=19.9237   9.1.1 The Liability Cap will [...]
  [2] score=0.6699    11.3. Liability Cap. Without [...] | [2] score=15.2422   9.1.2 The Liability Cap will [...]
  [3] score=0.6630    4.1.4 each Party’s [...]       | [3] score=14.2962   Section 9.10 Cap Collateral [...]

  Overlap in Top-3: 1 / 3 clauses

QUERY: 'termination for convenience notice period'


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


DENSE (Qdrant / MiniLM)                  BM25 (Okapi)                            
--------------------------------------------------------------------------------
  [1] score=0.7218    12.2 Termination for [...]     | [1] score=16.2942   15.2.5 Termination by Licensee [...]
  [2] score=0.6981    14.4 Termination By Datec For [...] | [2] score=15.7061   11.1 For Convenience. Customer [...]
  [3] score=0.6966    19.C. EXTENSION OF NOTICE. If [...] | [3] score=15.2174   7.5 Termination for [...]     

  Overlap in Top-3: 0 / 3 clauses



## 7. Summary Stats

In [13]:
import pandas as pd

df = pd.DataFrame(clauses)

print("=" * 50)
print("Knowledge Base Statistics")
print("=" * 50)
print(f"Total clauses     : {len(df):,}")
print(f"Unique contracts  : {df['contract_id'].nunique():,}")
print(f"Splits            : {df['split'].value_counts().to_dict()}")
print(f"\nClause length (chars):")
print(df['char_count'].describe().to_string())
print(f"\nVector DB         : Qdrant (cosine, dim={embeddings.shape[1]})")
print(f"BM25 index        : {BM25_INDEX_PATH.name}")
print(f"Embedding model   : {MODEL_NAME}")
print()
print("✅ Knowledge base is ready for the retriever fine-tuning stage!")

Knowledge Base Statistics
Total clauses     : 23,564
Unique contracts  : 511
Splits            : {'train': 23564}

Clause length (chars):
count     23564.000000
mean       1081.843490
std        3405.124724
min          81.000000
25%         246.000000
50%         466.000000
75%         922.000000
max      145582.000000

Vector DB         : Qdrant (cosine, dim=384)
BM25 index        : bm25_index.pkl
Embedding model   : sentence-transformers/all-MiniLM-L6-v2

✅ Knowledge base is ready for the retriever fine-tuning stage!
